# 01 · Dynamics, energy, and the Lie derivative

**Goal:** understand what the learned model is trying to establish before training it.
Run all cells on CPU; no checkpoint or solver is needed. You need basic matrix
algebra and derivatives. The code uses the same vector field as the EVT experiments.

For a state $x=(x_1,x_2)$, we study the **reverse-time** Van der Pol oscillator
$$\dot x_1=-x_2,\qquad \dot x_2=x_1+(x_1^2-1)x_2.$$
The origin is an equilibrium. Reversing the usual oscillator's time direction
makes the origin locally attracting. Its region of attraction is the set of
initial states whose trajectories approach the origin.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt

# Work from the repository root or anywhere inside it.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "tutorials/_support.py").is_file())
if str(ROOT / "tutorials") not in sys.path:
    sys.path.insert(0, str(ROOT / "tutorials"))
from _support import (setup, quadratic_matrix, warm_start, lie_derivative,
                      grid, sample_shell, VanDerPol)
rng = setup(42)
physics = VanDerPol(mu=1.0)
plt.rcParams.update({"figure.figsize": (6, 4), "axes.grid": False})
print("CPU tutorial | PyTorch", torch.__version__)

## 1. Look at the dynamics

Arrows describe instantaneous motion; trajectories integrate that motion.
We choose small initial states so these short integrations stay near the stable
origin. A finite trajectory plot cannot establish an entire region of attraction.

In [ ]:
from scipy.integrate import solve_ivp

X, Y, points = grid(limit=2.3, size=60)
flow = physics.vector_field(points).numpy()
fig, ax = plt.subplots()
ax.streamplot(X, Y, flow[:, 0].reshape(X.shape), flow[:, 1].reshape(Y.shape),
              color="0.75", density=0.9)
trajectories = []
for start in ([0.5, 0.0], [0.0, 0.8], [-0.8, -0.4]):
    sol = solve_ivp(lambda t, z: [-z[1], z[0] + (z[0]**2 - 1)*z[1]],
                    (0, 15), start, t_eval=np.linspace(0, 15, 300),
                    rtol=1e-8, atol=1e-10)
    assert sol.success and np.isfinite(sol.y).all()
    trajectories.append(sol)
    ax.plot(*sol.y, label=str(start))
ax.plot(0, 0, "ko")
ax.set(xlabel="$x_1$", ylabel="$x_2$", title="Reverse-time Van der Pol")
ax.legend(title="Initial state")
plt.show()

## 2. A candidate from the linearization

Near the origin, $f(x)\approx Ax$ with
$A=\begin{bmatrix}0&-1\\1&-1\end{bmatrix}$.
Solve $A^TP+PA=-I$ and define $V(x)=x^TPx$.
Positive eigenvalues of $P$ imply $V(0)=0$ and $V(x)>0$ for nonzero $x$.
For the **linearized** system, $\dot V=-\|x\|^2$ everywhere. The nonlinear
system contains a cubic term, so the same conclusion need not hold far away.

The helper calls SciPy's
[continuous Lyapunov solver](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.solve_continuous_lyapunov.html)
with `A.T` and `-I` to match this equation.

In [ ]:
P = quadratic_matrix()
A = torch.tensor([[0., -1.], [1., -1.]])
assert torch.allclose(A.T @ P + P @ A, -torch.eye(2), atol=1e-6)
assert torch.linalg.eigvalsh(P).min() > 0
print("P =", P.numpy(), "\neigenvalues =", torch.linalg.eigvalsh(P).numpy())

def quadratic(x):
    return torch.einsum("bi,ij,bj->b", x, P, x)

print("V(0) =", quadratic(torch.zeros(1, 2)).item())

## 3. Differentiate along the flow

The chain rule gives the **Lie derivative**
$$\dot V(x)=\nabla V(x)^T f(x).$$
PyTorch computes $\nabla V$ automatically. We also check it against a directional
finite difference, $[V(x+hf(x))-V(x-hf(x))]/(2h)$, holding $f(x)$ fixed.
This tests the calculation independently of its implementation.

In [ ]:
probe = torch.tensor([[0.3, 0.2], [-0.5, 0.4], [1.5, 1.0]])
value, derivative, _ = lie_derivative(quadratic, probe)
h = 1e-3
f = physics.vector_field(probe)
finite_difference = (quadratic(probe + h*f) - quadratic(probe - h*f)) / (2*h)
assert torch.allclose(derivative, finite_difference, atol=2e-3, rtol=2e-3)
print("Autograd:", derivative.detach().numpy())
print("Finite difference:", finite_difference.numpy())

X, Y, points = grid()
values, derivatives, _ = lie_derivative(quadratic, points)
V = values.detach().numpy().reshape(X.shape)
D = derivatives.detach().numpy().reshape(X.shape)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, field, title in zip(axes, [V, D], ["Quadratic V", "Lie derivative"]):
    image = ax.contourf(X, Y, field, levels=30, cmap="viridis" if title == "Quadratic V" else "coolwarm")
    fig.colorbar(image, ax=ax)
    ax.set(xlabel="$x_1$", ylabel="$x_2$", title=title)
axes[1].contour(X, Y, D, levels=[0], colors="black")
axes[1].contour(X, Y, V, levels=[0.2, 0.5, 1.0], colors="lime")
plt.tight_layout()
plt.show()

## 4. Sublevels and the logical gap

A sublevel $\Omega_\rho=\{x:V(x)\le\rho\}$ is a candidate ROA estimate.
For a smooth positive-definite $V$, strict decrease at every nonzero point in a
compact sublevel contained in the domain gives the usual Lyapunov argument for
attraction to the origin. Seeing negative values on a grid checks only that grid.
Also, the radius of a Euclidean ball and the value level $\rho$ are different.

In the plot, larger sublevels can enter regions with positive derivative. Training
will change the shape of $V$ to seek better sublevels. The next notebook explains
how positivity can hold by construction while decrease still has to be learned.

In [ ]:
fig, ax = plt.subplots()
for sol in trajectories:
    states = torch.tensor(sol.y.T, dtype=torch.float32)
    ax.plot(sol.t, quadratic(states).numpy())
ax.set(xlabel="Time", ylabel="$V(x(t))$", title="Candidate along the example trajectories")
plt.show()

## Try it

1. Replace `P` by the identity. Derive the new Lie derivative by hand and compare
   it with autograd. Is the derivative strictly negative everywhere near zero?
2. Increase the initial-state magnitudes. Does finite integration always converge?
3. Explain why a small simulation error and a dense grid are still not a proof.

**Next:** [02 — Train a candidate](02_train_a_candidate.ipynb).